# Repertoire Overlap Correctness Check

This notebook checks the correctness of the four repertoire-overlap implementations (SymDel, SymScan, XTNeighbor-streaming and CompAIRR), by comparing their outputs against real repertoires (from [Emerson et al](https://doi.org/10.1038/ng.3822)). It checks that the resulting repertoire x repertoire overlap matrices are identical, using CompAIRR as the reference.

Note: CompAIRR only supports indels (Levenshtein distance) when `d=1` (`d=2` errors out), so it is excluded from the `d=2` Levenshtein comparison; SymDel, SymScan and XTNeighbor-streaming are compared directly against each other in that case instead.

Run time: a few minutes, much shorter than the full benchmark since only small repertoire samples are needed to establish correctness.

## 0. Configuration

In [1]:
n_repeat = 1 # number of random repertoire subsets to test per configuration
sizes = [1, 2, 4, 8] # number of repertoires to sample
max_seqs_per_repertoire = 3000 # subsample each repertoire down to this many sequences, to keep CompAIRR's d=2 runs fast

## 1. Setup (run time ~ 3 min)

install dependency

In [2]:
! pip install -q pyrepseq

In [3]:
import os.path
import numpy as np
import pandas as pd

from airrutils import *

try:
    from google.colab import files
    colab = True
except ImportError:
    colab = False

In [4]:
import benchutils as bu

bu.describe_env()

{'colab': False,
 'platform': 'Linux-6.8.0-136-generic-x86_64-with-glibc2.39',
 'python': '3.12.13',
 'git_sha': '213f0ee',
 'cpu_model': '13th Gen Intel(R) Core(TM) i9-13900',
 'n_cpus_total': 32,
 'affinity': [0, 2, 4, 6, 8, 10, 12, 14],
 'n_cpus_visible': 8,
 'governor': 'performance',
 'no_turbo': '1',
 'gpu': {'name': 'NVIDIA GeForce RTX 4090',
  'memory_total': '24564 MiB',
  'clocks_max_sm': '3105 MHz',
  'clocks_applications_gr': '[N/A]'},
 'thread_env': {'RAYON_NUM_THREADS': '8',
  'OMP_NUM_THREADS': '8',
  'MKL_NUM_THREADS': '8',
  'OPENBLAS_NUM_THREADS': '8',
  'NUMEXPR_NUM_THREADS': '8',
  'NUMBA_NUM_THREADS': '8',
  'OMP_PROC_BIND': None,
  'OMP_PLACES': None},
 'packages': {'symscan': '0.8.3',
  'pyrepseq': '1.6',
  'pybktree': '1.1',
  'rapidfuzz': '3.14.5',
  'pwseqdist': '0.6',
  'numba': '0.66.0',
  'numpy': '2.4.6',
  'scipy': '1.18.0',
  'pandas': '3.0.5'},
 'timeout_seconds': 100}

clone the projects

In [5]:
if not os.path.exists("compairr"):
    !git clone https://github.com/uio-bmi/compairr.git

if colab and not os.path.exists("XT-neighbor"):
    !git clone https://github.com/heartnetkung/XT-neighbor.git
    repo_path = "XT-neighbor/"
else:
    repo_path = "../"
    
if not os.path.exists("symscan"):
    !git clone https://github.com/yutanagano/symscan.git

compile XTNeighbor-streaming

In [6]:
! mkdir -p {repo_path}xtneighbor_streaming/build
! cd {repo_path}xtneighbor_streaming/build; cmake ..;make

-- Configuring done (0.0s)
-- Generating done (0.0s)
-- Build files have been written to: /home/andreas/repos/XT-neighbor/xtneighbor_streaming/build


[100%] Built target xt_neighbor


compile Compairr

In [7]:
!cd compairr; make

make -C src compairr
make[1]: Entering directory '/home/andreas/repos/XT-neighbor/benchmarks/compairr/src'
make[1]: 'compairr' is up to date.
make[1]: Leaving directory '/home/andreas/repos/XT-neighbor/benchmarks/compairr/src'


install symscan-airr

In [8]:
!cd symscan; cargo install symscan-airr

    Updating crates.io index


     Ignored package `symscan-airr v0.1.0` is already installed, use --force to override


prepare repertoire info

In [9]:
def read_info():
  ans = pd.read_csv(f'{repo_path}/data/info_dl.csv')
  end = np.cumsum(ans['count'])
  ans['start'] = np.concatenate(([0],end[:-1]))
  ans['end'] = end
  return ans

info = read_info()
info

,file,count,start,end
0,HIP00110.tsv,96661,0,96661
1,HIP00169.tsv,90694,96661,187355
2,HIP00594.tsv,149386,187355,336741
3,HIP00602.tsv,179629,336741,516370
4,HIP00640.tsv,188327,516370,704697
...,...,...,...,...
695,Keck0109_MC1.tsv,155982,129798276,129954258
696,Keck0110_MC1.tsv,189472,129954258,130143730
697,Keck0111_MC1.tsv,242935,130143730,130386665
698,Keck0112_MC1.tsv,180685,130386665,130567350


In [10]:
! mkdir -p tmp

prepare input data

In [11]:
N_FILES=5

def read_input():
  for i in range(1,N_FILES+1):
    ! unzip -n {repo_path}/data/emerson_rep"$i"_dl.zip -d tmp
  reps = []
  for i in range(1,N_FILES+1):
    reps.append(pd.read_csv(f'tmp/emerson_rep{i}_dl.txt'))
  return pd.concat(reps,ignore_index=True)

data = read_input()
print(data.head())

Archive:  ..//data/emerson_rep1_dl.zip


Archive:  ..//data/emerson_rep2_dl.zip


Archive:  ..//data/emerson_rep3_dl.zip


Archive:  ..//data/emerson_rep4_dl.zip


Archive:  ..//data/emerson_rep5_dl.zip


                cdr3  count
0    CAAAAGGIAKNIQYF      1
1      CAAAEPSTDTQYF      1
2       CAAAGFNSPLHF      1
3  CAAAQDRGRVLGNEQFF      3
4    CAAAQGRSILDTQYF      7


check GPU availability

In [12]:
import subprocess
try:
  subprocess.run(["nvidia-smi"], capture_output=True, text=True)
except Exception as e:
  raise Exception("GPU required")

## 2. Correctness Check (run time < 2 minutes)

comparison harness: run every algorithm and compare its matrix against CompAIRR (or, when CompAIRR is unsupported, against whichever algorithm ran first)

In [13]:
algorithms = {
    'symdel': symdel_overlap,
    'symscan': lambda *a, **kw: symscan_airr(*a, **kw, return_matrix=True),
    'xt_streaming': lambda *a, **kw: xt_neighbor_overlap(*a, **kw, return_matrix=True),
    'compairr': lambda *a, **kw: compairr_overlap(*a, **kw, return_matrix=True),
}

def compare(distance, is_hamming, seqs, dup_counts, rep_sizes):
  matrices = {}
  for alg_name, fn in algorithms.items():
    result = fn(distance, is_hamming, seqs, dup_counts, rep_sizes)
    if result is not None:
      matrices[alg_name] = result

  ref_name = 'compairr' if 'compairr' in matrices else next(iter(matrices))
  ref = matrices[ref_name]

  mismatches = []
  for alg_name, mat in matrices.items():
    if alg_name == ref_name:
      continue
    ok = np.array_equal(mat, ref)
    print(f'    {alg_name:15s} vs {ref_name}: {"MATCH" if ok else "MISMATCH"}')
    if not ok:
      mismatches.append(alg_name)
  return mismatches


def run_exp(distance, is_hamming):
  measure = 'hamming' if is_hamming else 'leven'
  mismatches = set()
  for i in range(n_repeat):
    for size in sizes:
      seq_info, info_subset = sample_repertoire(data, info, size, random_state=i, max_seqs=max_seqs_per_repertoire)
      seqs, dup_counts, rep_sizes = prepare(seq_info, info_subset)
      print(f'distance={distance} measure={measure} n_repertoire={size} repeat={i}')
      mismatches.update(compare(distance, is_hamming, seqs, dup_counts, rep_sizes))
  return mismatches

In [14]:
mismatches = set()

In [15]:
mismatches |= run_exp(distance=1, is_hamming=True)

distance=1 measure=hamming n_repertoire=1 repeat=0


    symdel          vs compairr: MATCH
    symscan         vs compairr: MATCH
    xt_streaming    vs compairr: MATCH
distance=1 measure=hamming n_repertoire=2 repeat=0


    symdel          vs compairr: MATCH
    symscan         vs compairr: MATCH
    xt_streaming    vs compairr: MATCH
distance=1 measure=hamming n_repertoire=4 repeat=0


    symdel          vs compairr: MATCH
    symscan         vs compairr: MATCH
    xt_streaming    vs compairr: MATCH
distance=1 measure=hamming n_repertoire=8 repeat=0


    symdel          vs compairr: MATCH
    symscan         vs compairr: MATCH
    xt_streaming    vs compairr: MATCH


In [16]:
mismatches |= run_exp(distance=1, is_hamming=False)

distance=1 measure=leven n_repertoire=1 repeat=0


    symdel          vs compairr: MATCH
    symscan         vs compairr: MATCH
    xt_streaming    vs compairr: MATCH
distance=1 measure=leven n_repertoire=2 repeat=0


    symdel          vs compairr: MATCH
    symscan         vs compairr: MATCH
    xt_streaming    vs compairr: MATCH
distance=1 measure=leven n_repertoire=4 repeat=0


    symdel          vs compairr: MATCH
    symscan         vs compairr: MATCH
    xt_streaming    vs compairr: MATCH
distance=1 measure=leven n_repertoire=8 repeat=0


    symdel          vs compairr: MATCH
    symscan         vs compairr: MATCH
    xt_streaming    vs compairr: MATCH


In [17]:
mismatches |= run_exp(distance=2, is_hamming=True)

distance=2 measure=hamming n_repertoire=1 repeat=0


    symdel          vs compairr: MATCH
    symscan         vs compairr: MATCH
    xt_streaming    vs compairr: MATCH
distance=2 measure=hamming n_repertoire=2 repeat=0


    symdel          vs compairr: MATCH
    symscan         vs compairr: MATCH
    xt_streaming    vs compairr: MATCH
distance=2 measure=hamming n_repertoire=4 repeat=0


    symdel          vs compairr: MATCH
    symscan         vs compairr: MATCH
    xt_streaming    vs compairr: MATCH
distance=2 measure=hamming n_repertoire=8 repeat=0


    symdel          vs compairr: MATCH
    symscan         vs compairr: MATCH
    xt_streaming    vs compairr: MATCH


In [18]:
mismatches |= run_exp(distance=2, is_hamming=False)

distance=2 measure=leven n_repertoire=1 repeat=0


    symscan         vs symdel: MATCH
    xt_streaming    vs symdel: MATCH
distance=2 measure=leven n_repertoire=2 repeat=0


    symscan         vs symdel: MATCH
    xt_streaming    vs symdel: MATCH
distance=2 measure=leven n_repertoire=4 repeat=0


    symscan         vs symdel: MATCH
    xt_streaming    vs symdel: MATCH
distance=2 measure=leven n_repertoire=8 repeat=0


    symscan         vs symdel: MATCH
    xt_streaming    vs symdel: MATCH


In [19]:
if mismatches:
    raise Exception(f'comparison failed for: {sorted(mismatches)}')
print('success!')

success!
